# OOP basics
* until now we mostly wrote functions that work on data (lists, dicts, strings, ...) - the data and the logic that operates on it were always separate things
* object oriented programming (OOP) is a different way of structuring code: we bundle data (attributes) and the behaviour that belongs to that data (methods) together into one thing, called an object
* a "class" is the blueprint/template, an "object" (or "instance") is one concrete thing built from that blueprint - for example the class `Dog` describes what a dog generally is, while `my_dog = Dog("Rex")` is one specific dog
* remember from lesson 01: in python literally EVERYTHING already is an object - a string, an int, a list, even a function or a module. `type(42)` tells us `42` is an instance of the class `int`. we have been using OOP the whole time without writing our own classes yet, now its time to build our own
* this lesson is the first "real" OOP lesson - lesson 06 already sneak-peeked classes a tiny bit when we built a custom exception class, we will fully understand what happened there by the end of this lesson

## defining a class
* a class is defined with the keyword `class`, followed by a name (convention: CapitalizedWords, also called "CamelCase") and a `:`
* `__init__` is a special method (a "constructor") that python calls automatically every time we create a new instance - it sets up the initial state of the object
* `self` is the first parameter of every instance method, it refers to "this specific object" - python passes it in automatically, we never pass it ourselves when calling the method

In [1]:
# a minimal class definition
class Dog:
    # __init__ runs automatically when we create a new Dog(...)
    # "self" always refers to the object currently being created/used - python fills it in for us
    def __init__(self, name, age):
        self.name = name  # store the given name ON this specific object (an "instance attribute")
        self.age = age    # same for age

# creating an instance ("instantiating" the class) - this calls __init__ behind the scenes
my_dog = Dog("Rex", 3)     # self is filled automatically, we only pass name and age
other_dog = Dog("Fido", 5)  # a second, completely independent object

print(my_dog.name, my_dog.age)      # accessing instance attributes with the dot "."
print(other_dog.name, other_dog.age)

# each instance has its own separate copy of the attributes
my_dog.age = 4
print(my_dog.age, other_dog.age)  # only my_dog changed, other_dog is untouched

# type() confirms my_dog really is an instance of our own class, same mechanism as type(42) is int
print(type(my_dog))
print(isinstance(my_dog, Dog))  # isinstance checks if an object belongs to a class (or a subclass of it)

Rex 3
Fido 5
4 5
<class '__main__.Dog'>
True


## instance attributes vs class attributes
* instance attributes (like `self.name` above) belong to ONE specific object, every instance gets its own copy
* class attributes are defined directly inside the class body (not inside `__init__`) and are SHARED by all instances of that class
* classic beginner pitfall: never use a mutable object (list, dict, set) as a class attribute if you actually want each instance to have its own independent one - since it is shared, changing it through one instance affects all of them!

In [2]:
class Employee:
    company_name = "Acme Corp"  # class attribute - lives on the CLASS itself, shared by every instance

    def __init__(self, name):
        self.name = name  # instance attribute - each employee gets their own name

alice = Employee("Alice")
bob = Employee("Bob")

# both instances see the same class attribute
print(alice.company_name, bob.company_name)

# changing the class attribute through the CLASS changes it for everyone
Employee.company_name = "New Corp Name"
print(alice.company_name, bob.company_name)  # both updated!

# but assigning to it through an INSTANCE just creates a new instance attribute that shadows the class one
alice.company_name = "Alice's Own Startup"
print(alice.company_name, bob.company_name)  # only alice changed now, bob still sees the class attribute


# now the classic pitfall: a MUTABLE class attribute (a list here)
class ShoppingCart:
    items = []  # DANGER: this list is created ONCE and shared by ALL instances, not what we usually want!

    def add_item(self, item):
        self.items.append(item)  # this mutates the SHARED list, not a per-instance one

cart_a = ShoppingCart()
cart_b = ShoppingCart()

cart_a.add_item("apple")
print(cart_a.items)  # ['apple'] - looks fine so far
print(cart_b.items)  # ['apple'] too?! cart_b never added anything, but shares the same list object!

# the fix: create mutable defaults INSIDE __init__ instead, so every instance gets its OWN fresh list
class ShoppingCartFixed:
    def __init__(self):
        self.items = []  # a brand new list, created fresh for every single instance

    def add_item(self, item):
        self.items.append(item)

cart_c = ShoppingCartFixed()
cart_d = ShoppingCartFixed()
cart_c.add_item("banana")
print(cart_c.items)  # ['banana']
print(cart_d.items)  # [] - correctly independent this time!
# this is the exact same "mutable default" trap we saw with function default arguments in earlier lessons

Acme Corp Acme Corp
New Corp Name New Corp Name
Alice's Own Startup New Corp Name
['apple']
['apple']
['banana']
[]


## instance methods
* a method is just a function defined inside a class - the only real difference is it always receives `self` as its first parameter
* through `self` a method can read and modify the attributes of the specific instance it was called on
* we call a method with `object.method(arguments)` - python automatically passes the object itself as `self`, we only supply the remaining arguments

In [3]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    # an instance method - "self" gives us access to THIS dog's own attributes
    def bark(self):
        print(f"{self.name} says woof!")

    # methods can also take extra parameters, just like normal functions (after self)
    def have_birthday(self):
        self.age += 1  # modifies the attribute of this specific instance
        print(f"{self.name} is now {self.age} years old")

    # methods can call OTHER methods of the same object, again through self
    def celebrate(self):
        self.bark()          # self.bark() is equivalent to Dog.bark(self)
        self.have_birthday()

rex = Dog("Rex", 3)
rex.bark()             # python translates this call into Dog.bark(rex) behind the scenes
rex.have_birthday()
rex.celebrate()

fido = Dog("Fido", 5)
fido.bark()  # fido has its own independent state, calling bark() on it never touches rex

Rex says woof!
Rex is now 4 years old
Rex says woof!
Rex is now 5 years old
Fido says woof!


## `__str__` vs `__repr__`
* these are two "dunder" (double underscore) / "magic" methods that control how an object turns into text
* `__str__` should return a nice, readable, human-friendly text representation - python calls it for `print(obj)`, `str(obj)` and f-strings
* `__repr__` should return an unambiguous, more technical representation, ideally one that could be used to recreate the object - python calls it for `repr(obj)` and for the interactive echo (just typing `obj` in a REPL/notebook cell without print)
* if `__str__` is missing, python falls back to using `__repr__` instead - so defining at least `__repr__` is generally a good habit

In [4]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # the "friendly" version, meant for humans - used by print() and str()
        return f"({self.x}, {self.y})"

    def __repr__(self):
        # the "technical/debug" version - used by repr() and the notebook/REPL echo
        # convention: try to make it look like valid python code that recreates the object
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 4)

print(p)        # calls __str__ --> (3, 4)
print(str(p))   # also calls __str__
print(repr(p))  # calls __repr__ --> Point(x=3, y=4)

p  # just echoing the object with no print - notebooks/REPL use __repr__ for this

# lists always use repr() for their elements, even inside print() - handy to know!
print([p, Point(0, 0)])

(3, 4)
(3, 4)
Point(x=3, y=4)
[Point(x=3, y=4), Point(x=0, y=0)]


## more dunder / magic methods: operator overloading
* python lets us define what the built-in operators (`==`, `+`, `len()`, ...) should DO for our own classes, by implementing the matching dunder method
* `__eq__(self, other)` defines what `==` does (without it, `==` just checks if its literally the same object in memory)
* `__len__(self)` defines what `len(obj)` returns
* `__add__(self, other)` defines what `+` does between two objects of our class - this is called "operator overloading"
* there are many more (`__sub__`, `__lt__`, `__contains__`, ...) - we only cover the most common ones here

In [5]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    # without __eq__, "==" would fall back to comparing identity (is it the SAME object?)
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y  # we define "equal" as "same coordinates"

    # lets pretend a Point's "length" is how many coordinates it has - just to demo __len__, always 2 here
    def __len__(self):
        return 2

    # "+" between two points adds their coordinates and returns a brand new Point
    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

a = Point(1, 2)
b = Point(1, 2)
c = Point(5, 5)

print(a == b)   # True - same coordinates, thanks to our __eq__
print(a == c)   # False
print(a is b)   # False! "is" checks identity (same object in memory), completely different from "=="

print(len(a))   # 2 - calls our __len__

print(a + c)    # calls our __add__, returns a new Point(6, 7)
print(a)        # a itself is unchanged, __add__ returned a NEW object, it didnt modify a or c

True
False
False
2
Point(x=6, y=7)
Point(x=1, y=2)


## `@staticmethod` and `@classmethod`
* a normal instance method always gets `self` (the specific object) automatically as its first parameter
* a `@staticmethod` gets NEITHER `self` NOR the class automatically - its basically just a normal function that happens to live inside the class namespace, usually because it is closely related to the class conceptually
* a `@classmethod` gets the CLASS itself automatically as its first parameter (conventionally named `cls`, not `self`) - a very common use case is an "alternative constructor": a different way to build an instance than the default `__init__`

In [6]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    # a staticmethod - no self, no cls, just a plain helper function grouped inside the class
    @staticmethod
    def distance_between(point_a, point_b):
        # pure calculation, doesnt need any specific instance's state
        return ((point_a.x - point_b.x) ** 2 + (point_a.y - point_b.y) ** 2) ** 0.5

    # a classmethod - gets the CLASS itself as "cls", used here as an "alternative constructor"
    @classmethod
    def from_tuple(cls, coordinates):
        # cls refers to Point here - calling cls(...) is the same as calling Point(...)
        # this lets subclasses reuse this constructor correctly too (cls adapts, "Point" hardcoded wouldnt)
        x, y = coordinates
        return cls(x, y)

# calling a staticmethod - through the class or through an instance, both work the same, no self/cls needed
print(Point.distance_between(Point(0, 0), Point(3, 4)))  # 5.0

# calling the classmethod alternative constructor - reads almost like a sentence: "make a Point from a tuple"
p = Point.from_tuple((7, 8))
print(p)

# compare to the normal constructor - from_tuple is just a more convenient/readable entry point for this use case
p2 = Point(7, 8)
# this Point class (redefined fresh in this cell) has no __eq__ of its own, so "==" falls back to the default
# from the builtin object class: identity comparison ("is this the exact same object in memory?")
# p and p2 are two DIFFERENT objects that just happen to hold the same x/y values, so this is False
# we will build our own __eq__ properly in the very next section
print(p == p2)

5.0
Point(x=7, y=8)
False


## `@property` - computed attributes
* a property lets a METHOD be accessed like a plain attribute (no parentheses needed) - useful for values that are computed on the fly from other attributes
* a property setter lets us run validation logic whenever someone tries to ASSIGN a new value - this way we can reject invalid data, while the calling code still just looks like a normal attribute assignment

In [7]:
class Circle:
    def __init__(self, radius):
        self.radius = radius  # this actually calls the setter below already, because of the property!

    @property
    def area(self):
        # this looks like a plain attribute from the outside (circle.area, no brackets) but is computed every time
        return 3.14159 * self.radius ** 2

    @property
    def radius(self):
        return self._radius  # the real value is stored in a "private-ish" _radius (more on the underscore soon)

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("radius cant be negative")  # validation happens right here, on every assignment
        self._radius = value

circle = Circle(5)
print(circle.radius)  # looks like a normal attribute access, but actually calls our @property getter method
print(circle.area)    # computed fresh every time - no parentheses needed even though it runs code!

circle.radius = 10    # looks like a normal assignment, but actually calls our @radius.setter method
print(circle.area)    # area automatically reflects the new radius, since it recomputes every access

try:
    circle.radius = -3  # this triggers our validation inside the setter
except ValueError as error:
    print("error:", error)

5
78.53975
314.159
error: radius cant be negative


## encapsulation conventions
* encapsulation means hiding the internal details of an object and only exposing a clean interface - python does this mostly by CONVENTION, not by strict enforcement
* public (`name`) - normal attribute, meant to be used freely from outside the class
* `_protected` (single leading underscore) - a convention meaning "internal use, please dont touch this from outside", but python does NOT actually stop you, its just a polite warning to other developers
* `__private` (double leading underscore) - triggers "name mangling": python internally renames it to `_ClassName__attributename`, making it awkward (but still not impossible!) to access from outside

In [8]:
class Account:
    def __init__(self, balance):
        self.balance = balance        # public - fair game to use directly
        self._pin = "1234"            # protected by convention only - "please dont touch me", nothing enforces it
        self.__secret_key = "xyz789"  # name-mangled - python actively rewrites this attribute name

account = Account(100)
print(account.balance)   # totally fine, its public
print(account._pin)      # also technically works! python does NOT prevent this, its purely a naming convention

# lets see what happened to __secret_key with vars() (shows all instance attributes as a dict)
print(vars(account))
# notice "__secret_key" became "_Account__secret_key" - this is the "name mangling" in action

# so accessing account.__secret_key directly fails...
try:
    print(account.__secret_key)
except AttributeError as error:
    print("error:", error)

# ...but the mangled name still works, proving its convention/obfuscation, not real enforced privacy like Java's private
print(account._Account__secret_key)

# dir() also confirms the mangled name is what actually exists on the object
print([name for name in dir(account) if "secret" in name.lower()])

100
1234
{'balance': 100, '_pin': '1234', '_Account__secret_key': 'xyz789'}
error: 'Account' object has no attribute '__secret_key'
xyz789
['_Account__secret_key']


## putting it all together: a `BankAccount` class
* lets combine what we learned into one small, realistic example: instance attributes, a method, a `@property` with validation, `__str__`, and raising an exception (like we learned in lesson 06) when something goes wrong

In [9]:
class InsufficientFundsError(Exception):
    # a custom exception, exactly like the sneak-peek we saw back in lesson 06
    pass

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance  # stored "privately" (by convention), exposed through the property below

    @property
    def balance(self):
        return self._balance  # read access - just returns the current balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance cant be set to a negative number directly")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit amount must be positive")  # basic validation
        self._balance += amount  # we modify the underlying attribute directly here, we already validated ourselves

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdraw amount must be positive")
        if amount > self._balance:
            # a custom exception communicates exactly WHAT domain problem happened, better than a generic error
            raise InsufficientFundsError(f"cant withdraw {amount}, balance is only {self._balance}")
        self._balance -= amount

    def __str__(self):
        return f"BankAccount(owner={self.owner!r}, balance={self._balance})"

acc = BankAccount("Alex", 100)
print(acc)  # uses our __str__

acc.deposit(50)
print(acc.balance)  # 150, read through the property

acc.withdraw(30)
print(acc.balance)  # 120

try:
    acc.withdraw(1000)  # way more than the balance
except InsufficientFundsError as error:
    print("caught:", error)

try:
    acc.balance = -5  # our property setter validation kicks in
except ValueError as error:
    print("caught:", error)

print(acc)  # still shows the balance from before the failed operations, both raises left it untouched

BankAccount(owner='Alex', balance=100)
150
120
caught: cant withdraw 1000, balance is only 120
caught: balance cant be set to a negative number directly
BankAccount(owner='Alex', balance=120)


## quick comparison to other languages
* python methods take `self` as an EXPLICIT first parameter that we write ourselves - Java, C#, C++ and JavaScript have an implicit `this` that is just automatically available inside a method, never written as a parameter
* python's constructor is always named `__init__` regardless of the class name - Java and C# name the constructor exactly like the class (`public BankAccount(...)`), C++ does the same, JavaScript uses a method literally called `constructor(...)` inside the class body
* python's `_protected` and `__private` (name mangled) are pure NAMING CONVENTIONS - nothing stops determined code from accessing them anyway. Java, C# and C++ have real `private`/`protected` keywords that are enforced by the compiler, breaking the build if violated
* python's `@property` lets a method be called like a plain attribute with custom getter/setter logic hidden behind normal attribute syntax. C# has very similar built-in `get`/`set` property syntax at the language level. Java has no property syntax at all - the convention is writing manual `getBalance()`/`setBalance()` methods. JavaScript (ES6+) has `get`/`set` keywords inside a class, conceptually close to python's `@property`
* `@staticmethod` corresponds to `static` methods in Java/C#/C++/JS - a method that belongs to the class but not to any specific instance
* python has no direct `@classmethod` equivalent in most of those languages - Java/C#/C++ static methods cant receive "which class was this really called on" the way `cls` can, which matters for correct inheritance in alternative constructors

## Exercises
1. write a class `Rectangle` with `__init__(self, width, height)`, a `@property` called `area` that computes `width * height`, and a `__str__` that returns something like `"Rectangle(3 x 4)"`
2. write a class `Temperature` with `__init__(self, celsius)` storing the value, and a `@classmethod` `from_fahrenheit(cls, fahrenheit)` that creates a `Temperature` by converting fahrenheit to celsius (`celsius = (fahrenheit - 32) * 5 / 9`)
3. write a class `Vector` with `__init__(self, x, y)`, `__add__` (returns a new `Vector` with added coordinates), `__eq__` (compares x and y) and `__repr__` returning something like `"Vector(x=1, y=2)"`
4. write a class `Stack` with a private list attribute (use `__` name mangling), and methods `push(self, item)`, `pop(self)` (removes and returns the last item, raise a custom exception `EmptyStackError` if the stack is empty) and `__len__` returning how many items are currently on the stack

In [10]:
# TODO: solve exercise 1 here (Rectangle with area property and __str__)


# TODO: solve exercise 2 here (Temperature with from_fahrenheit classmethod)


# TODO: solve exercise 3 here (Vector with __add__, __eq__, __repr__)


# TODO: solve exercise 4 here (Stack with name-mangled list, push/pop/__len__, EmptyStackError)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [11]:
# sample solution 1 - Rectangle
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height  # computed fresh every access, always in sync with width/height

    def __str__(self):
        return f"Rectangle({self.width} x {self.height})"

rect = Rectangle(3, 4)
print(rect)
print(rect.area)


# sample solution 2 - Temperature with alternative constructor
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        celsius = (fahrenheit - 32) * 5 / 9  # convert, then let the normal constructor take over
        return cls(celsius)

t = Temperature.from_fahrenheit(98.6)
print(round(t.celsius, 1))  # about 37.0 - normal human body temperature


# sample solution 3 - Vector
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)  # returns a brand new Vector

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __repr__(self):
        return f"Vector(x={self.x}, y={self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)               # Vector(x=4, y=6)
print(v1 == Vector(1, 2))    # True


# sample solution 4 - Stack
class EmptyStackError(Exception):
    pass

class Stack:
    def __init__(self):
        self.__items = []  # name-mangled to _Stack__items, "private by convention" storage

    def push(self, item):
        self.__items.append(item)

    def pop(self):
        if not self.__items:  # empty list is falsy, remember from lesson 03!
            raise EmptyStackError("cant pop from an empty stack")
        return self.__items.pop()

    def __len__(self):
        return len(self.__items)

stack = Stack()
stack.push(1)
stack.push(2)
stack.push(3)
print(len(stack))   # 3
print(stack.pop())  # 3, last in first out
print(len(stack))   # 2

empty_stack = Stack()
try:
    empty_stack.pop()
except EmptyStackError as error:
    print("caught:", error)

Rectangle(3 x 4)
12
37.0
Vector(x=4, y=6)
True
3
3
2
caught: cant pop from an empty stack


* congratulation you finished this lesson

## what we learned
* what classes, objects/instances are, and that everything in python already is an object
* defining a class with `class`, the `__init__` constructor, `self`, instance attributes vs. shared class attributes and the mutable-default-class-attribute pitfall
* instance methods and how `self` gives access to that specific object's state
* `__str__` (human friendly, used by print/str) vs `__repr__` (technical, used by repr/echo), and other dunder methods for operator overloading (`__eq__`, `__len__`, `__add__`)
* `@staticmethod` (no self/cls) vs `@classmethod` (gets `cls`, great for alternative constructors) vs normal instance methods
* `@property` for computed, attribute-like access, and property setters for validation on assignment
* encapsulation conventions: public, `_protected` (convention only), `__private` (real name mangling), and that python enforces none of this the way Java/C#/C++ do
* combined everything into a small `BankAccount` example with validation and a custom exception